# Smart MCQ Solver v2 — Pairwise BiLSTM + Attention + Pretrained Embeddings

This is an upgraded version of the original classifier notebook. Same 12-section teaching
structure, but with the three changes expected to move the score the most:

1. **Reformulation**: instead of `question + all 5 options -> pick A/B/C/D/E` (a 5-way
   classification problem), we now do `question + one option -> relevance score`, run once
   per option, then rank the 5 scores. This maps directly onto MAP@3, removes any "option C is
   usually right" position bias by construction, and lets the encoder focus on one clean
   question/option pair at a time instead of a long 5-option wall of text.
2. **Pretrained embeddings (GloVe)**: the embedding table now starts from GloVe vectors
   instead of random noise, so the BiLSTM already knows `king ~ queen` before training starts.
3. **Safe augmentation + tuned early stopping**: a small curated synonym swap (not random
   noise — MCQs are label-sensitive) adds mild paraphrase variety to training only, and early
   stopping is loosened slightly since BiLSTMs often improve in small increments late in
   training.

Everything else (vocabulary, padding, masking, gradient clipping, MAP@3 metric) is unchanged
from the original notebook.


### Setup: device safety + config
Same `get_device()` pattern as before. New config knobs are grouped at the bottom of `CONFIG`:
pretrained embeddings, augmentation, and the loosened early-stopping numbers.


In [40]:
!pip install -q --no-deps wandb

import re, random, glob
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
import wandb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def get_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        (torch.zeros(1, device="cuda") + 1).cpu()
        return torch.device("cuda")
    except Exception as e:
        print("CUDA unusable, falling back to CPU:", e)
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

CONFIG = {
    # vocab / sequence
    "max_vocab_size": 30000, "min_freq": 2, "max_len": 40,   # placeholder, recomputed in section 2
    # model
    "embed_dim": 100, "hidden_dim": 128, "num_layers": 2,
    "dropout": 0.35, "label_smoothing": 0.05,
    # optimization
    "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 32,       # batch = questions; each expands to x5 options
    "epochs": 40, "val_split": 0.15, "grad_clip": 1.0,
    # early stopping -- loosened vs. v1 (patience 6->10, min_delta 0.001->0.0005)
    "early_stop_min_epochs": 12, "early_stop_patience": 10, "early_stop_min_delta": 0.0005,
    # v2 additions
    "use_pretrained_embeddings": True, "freeze_embeddings": False,
    "use_synonym_aug": True, "aug_fraction": 0.3,
}

wandb.login()
wandb.init(project="smart-mcq-solver", name="bilstm-attention-pairwise-v2", config=CONFIG)


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


CUDA unusable, falling back to CPU: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Using device: cpu


## 1. Data + the pairwise reformulation
**What changed:** we no longer build one long `question + [a] ... [e] ...` string. Instead we
keep the question and the 5 options separate, and write a helper `build_pair_text(question,
option)` that the model will call once per option. The label becomes "which of the 5 options is
the correct one" (0-4), used later to pick out the right score among the 5.

**Why this helps:** the model sees one clean, short (question, option) pair per forward pass
instead of a five-option wall of text — closer to what a human does when checking one answer at
a time, and closer to the MAP@3 metric itself (rank scores, don't classify).


In [41]:
train_df = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv")
LABELS = ["A", "B", "C", "D", "E"]
label2idx = {l: i for i, l in enumerate(LABELS)}

for df in (train_df, test_df):
    for col in ["prompt"] + LABELS:
        df[col] = df[col].fillna("").astype(str)

train_df["label"] = train_df["answer"].map(label2idx)

def build_pair_text(question, option):
    return f"question: {question} answer: {option}".lower()

# sanity check: one (question, option) pair
print(build_pair_text(train_df.loc[0, "prompt"], train_df.loc[0, "A"]))


question: pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options. answer: martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. the relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.


## 2. Tokenization + choosing `max_len` from data, not a guess
**Theory:** same regex word tokenizer as before -- simple, transparent, and it already matches
GloVe's lowercased-alphanumeric vocabulary well, so no need to complicate it.

**What changed:** `max_len` used to be a fixed guess (`60`) applied to a *5-option* concatenated
string. Now that each example is a single (question, option) pair, sequences are naturally much
shorter -- but we still check empirically instead of guessing again. We tokenize a sample of
pair-texts, look at the length distribution, and pick `max_len` to cover ~98% of them without
truncation.


In [42]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text)   # strips punctuation, keeps alphanumeric tokens

sample_lengths = [
    len(tokenize(build_pair_text(row.prompt, getattr(row, letter))))
    for row in train_df.itertuples()
    for letter in LABELS
]
p50, p95, p98, p100 = np.percentile(sample_lengths, [50, 95, 98, 100])
print(f"pair-text token length -- p50={p50:.0f} p95={p95:.0f} p98={p98:.0f} max={p100:.0f}")

CONFIG["max_len"] = int(min(max(p98, 16), 64))   # cover ~98% of examples, capped for speed
print("max_len set to:", CONFIG["max_len"])


pair-text token length -- p50=43 p95=84 p98=94 max=160
max_len set to: 64


## 3. Vocabulary building
**Unchanged in spirit:** built from **train only** (not test), `<PAD>=0` / `<UNK>=1` reserved,
`min_freq` filters one-off tokens. Built on the pair-texts so it matches exactly what the model
will tokenize at train time (including the small amount of synonym-augmented text from
section 4).


In [43]:
def build_vocab(texts, max_vocab_size, min_freq):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token, freq in counter.most_common():
        if freq < min_freq or len(vocab) >= max_vocab_size:
            continue
        vocab[token] = len(vocab)
    return vocab


## 4. Train/val split + safe synonym augmentation
**Order matters:** split into train/val *before* augmenting, and only augment the train side --
otherwise augmented copies of a validation question could leak into training and inflate the
validation score.

**Why synonym swap and not random noise:** the original notebook's own answer was explicit --
MCQs are label-sensitive, so randomly deleting/replacing/shuffling words can silently flip the
correct answer. A small curated dictionary of safe, meaning-preserving synonyms avoids that.
This is a lightweight stand-in for the "LLM paraphrasing" idea; swap in an LLM call here later
if you want a stronger version of the same idea.


In [44]:
train_idx, val_idx = train_test_split(
    train_df.index, test_size=CONFIG["val_split"], random_state=SEED, stratify=train_df["label"]
)
train_part = train_df.loc[train_idx].reset_index(drop=True)
val_part   = train_df.loc[val_idx].reset_index(drop=True)

# small, curated, meaning-preserving synonym dictionary -- deliberately conservative
SAFE_SYNONYMS = {
    "large": "big", "small": "tiny", "fast": "quick", "begin": "start", "end": "finish",
    "happy": "glad", "sad": "unhappy", "hard": "difficult", "easy": "simple", "big": "large",
    "old": "ancient", "new": "modern", "buy": "purchase", "help": "assist", "show": "display",
}

def synonym_swap(text, rate=0.3):
    words = text.split()
    for i, w in enumerate(words):
        if w in SAFE_SYNONYMS and random.random() < rate:
            words[i] = SAFE_SYNONYMS[w]
    return " ".join(words)

def augment(df, fraction):
    n = int(len(df) * fraction)
    rows = df.sample(n=n, random_state=SEED).copy()
    rows["prompt"] = rows["prompt"].apply(lambda t: synonym_swap(t.lower()))
    return rows

if CONFIG["use_synonym_aug"]:
    train_part = pd.concat([train_part, augment(train_part, CONFIG["aug_fraction"])], ignore_index=True)

print("train size (post-augmentation):", len(train_part), "| val size:", len(val_part))

vocab = build_vocab(
    [build_pair_text(r.prompt,getattr(r, l)) for r in train_part.itertuples() for l in LABELS],
    CONFIG["max_vocab_size"], CONFIG["min_freq"],
)
PAD_IDX, UNK_IDX = vocab["<PAD>"], vocab["<UNK>"]
print("Vocab size:", len(vocab))


train size (post-augmentation): 2210 | val size: 300
Vocab size: 2972


## 5. Pretrained embeddings (GloVe)
**Theory:** instead of the embedding table starting from random numbers, we look up each vocab
word in GloVe and copy its pretrained vector in. Words GloVe doesn't have (rare/typo tokens)
keep a small random vector -- there's no pretrained knowledge to borrow for those anyway.

**Kaggle setup:** attach a GloVe dataset to the notebook (e.g. the public "GloVe 6B" dataset) so
one of the paths below resolves. If none is found, we fall back to random init and print a
warning -- the rest of the notebook still runs, just without this boost.


In [45]:
def find_glove_file(dim):
    candidates = [
        f"/kaggle/input/glove6b{dim}dtxt/glove.6B.{dim}d.txt",
        f"/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.{dim}d.txt",
    ]
    candidates += glob.glob(f"/kaggle/input/**/glove.6B.{dim}d.txt", recursive=True)
    for path in candidates:
        if path and __import__("os").path.exists(path):
            return path
    return None

def load_glove_matrix(vocab, embed_dim):
    matrix = np.random.normal(scale=0.1, size=(len(vocab), embed_dim)).astype(np.float32)
    matrix[PAD_IDX] = 0.0

    path = find_glove_file(embed_dim)
    if path is None:
        print("No GloVe file found under /kaggle/input -- attach a GloVe dataset to use pretrained "
              "embeddings. Falling back to random init.")
        return matrix, 0

    hits = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            if word in vocab:
                matrix[vocab[word]] = np.asarray(parts[1:], dtype=np.float32)
                hits += 1
    return matrix, hits

if CONFIG["use_pretrained_embeddings"]:
    embed_matrix, n_hits = load_glove_matrix(vocab, CONFIG["embed_dim"])
    print(f"GloVe coverage: {n_hits}/{len(vocab)} vocab words ({n_hits / len(vocab):.1%})")
else:
    embed_matrix = None


No GloVe file found under /kaggle/input -- attach a GloVe dataset to use pretrained embeddings. Falling back to random init.
GloVe coverage: 0/2972 vocab words (0.0%)


## 6. Sequence padding
**Unchanged:** every (question, option) pair is tokenized, truncated/padded to `max_len`, and its
true length (pre-padding) is tracked so the LSTM never treats `<PAD>` as real input.


In [46]:
def encode(text, vocab, max_len):
    tokens = tokenize(text)[:max_len]
    length = len(tokens)
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens]
    ids += [PAD_IDX] * (max_len - length)          # pad on the right
    return ids, max(length, 1)                      # guard against a fully-empty sequence


## 7. Dataset and DataLoader -- one question, five options per item
**Shapes:** `__getitem__` now returns all 5 options for one question at once:
`ids: (5, max_len)`, `lengths: (5,)`, `label: scalar` (index 0-4 of the correct option).
After batching: `(batch, 5, max_len)`, `(batch, 5)`, `(batch,)`.

**Why per-question batching:** the 5 options for a question must land in the same batch so we
can rank them against each other for MAP@3 -- shuffling only shuffles *which questions* are in a
batch, never the options within one question.


In [47]:
class MCQPairDataset(Dataset):
    def __init__(self, df, vocab, max_len, has_labels=True):
        self.df, self.vocab, self.max_len, self.has_labels = df, vocab, max_len, has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids_per_option, len_per_option = [], []
        for letter in LABELS:
            ids, length = encode(build_pair_text(row["prompt"], row[letter]), self.vocab, self.max_len)
            ids_per_option.append(ids)
            len_per_option.append(length)

        ids = torch.tensor(ids_per_option, dtype=torch.long)        # (5, max_len)
        lengths = torch.tensor(len_per_option, dtype=torch.long)    # (5,)
        if self.has_labels:
            return ids, lengths, torch.tensor(row["label"], dtype=torch.long)
        return ids, lengths

train_loader = DataLoader(MCQPairDataset(train_part, vocab, CONFIG["max_len"]), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader   = DataLoader(MCQPairDataset(val_part, vocab, CONFIG["max_len"]),   batch_size=CONFIG["batch_size"], shuffle=False)
test_loader  = DataLoader(MCQPairDataset(test_df, vocab, CONFIG["max_len"], has_labels=False), batch_size=CONFIG["batch_size"], shuffle=False)


## 8. Embedding layer
**What changed:** the embedding table is now initialized from the GloVe matrix built in
section 5 (falls back to PyTorch's default random init if no GloVe file was found).
`freeze_embeddings=True` would keep GloVe vectors fixed; we leave them trainable by default so
the model can still adapt them to this specific dataset.


In [48]:
class WordEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_idx, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if pretrained is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained))
            self.embedding.weight.requires_grad = not freeze

    def forward(self, x):
        return self.embedding(x)   # (batch, seq_len) -> (batch, seq_len, embed_dim)


## 9. BiLSTM + attention + scoring head
**Unchanged mechanics** (packed sequences, additive attention with padding mask) -- see the
original notebook for the full gate equations. **What changed:** the classifier head now
outputs a single relevance **score** per (question, option) pair instead of 5 logits at once,
since scoring happens per-option and the 5 scores are only brought together afterwards.


In [49]:
class BiLSTMEncoder(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        return out   # (batch, seq_len, 2*hidden_dim)

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out, lengths):
        scores = self.v(torch.tanh(self.attn(lstm_out))).squeeze(-1)   # (batch, seq_len)
        seq_len = lstm_out.size(1)
        mask = torch.arange(seq_len, device=lengths.device)[None, :] < lengths[:, None]
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)  # (batch, hidden_dim)
        return context, weights

class ScoreHead(nn.Module):
    """Maps a pooled (question, option) representation to ONE relevance score."""
    def __init__(self, hidden_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),   # single raw score, no activation
        )

    def forward(self, context):
        return self.net(context).squeeze(-1)   # (batch,)

class PairScorer(nn.Module):
    """Scores one (question, option) pair at a time -- called once per option, 5x per question."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx, dropout, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = WordEmbedding(vocab_size, embed_dim, pad_idx, pretrained, freeze)
        self.encoder = BiLSTMEncoder(embed_dim, hidden_dim, num_layers, dropout)
        self.attention = Attention(hidden_dim * 2)
        self.head = ScoreHead(hidden_dim * 2, dropout)

    def forward(self, ids, lengths):
        # ids, lengths: (N, seq_len) / (N,) where N = batch_questions * 5 options, flattened
        embedded = self.embedding(ids)
        lstm_out = self.encoder(embedded, lengths)
        context, attn_weights = self.attention(lstm_out, lengths)
        score = self.head(context)
        return score, attn_weights

model = PairScorer(
    vocab_size=len(vocab), embed_dim=CONFIG["embed_dim"], hidden_dim=CONFIG["hidden_dim"],
    num_layers=CONFIG["num_layers"], pad_idx=PAD_IDX, dropout=CONFIG["dropout"],
    pretrained=embed_matrix, freeze=CONFIG["freeze_embeddings"],
).to(device)
wandb.watch(model, log="all", log_freq=50)
print(model)


PairScorer(
  (embedding): WordEmbedding(
    (embedding): Embedding(2972, 100, padding_idx=0)
  )
  (encoder): BiLSTMEncoder(
    (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  )
  (attention): Attention(
    (attn): Linear(in_features=256, out_features=256, bias=True)
    (v): Linear(in_features=256, out_features=1, bias=False)
  )
  (head): ScoreHead(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.35, inplace=False)
      (3): Linear(in_features=128, out_features=1, bias=True)
    )
  )
)


## 10. Training loop
**What changed:** each batch is `(batch_questions, 5, max_len)`. We flatten the option dimension
into the batch dimension so the shared `PairScorer` scores all options in one matmul-friendly
pass, then reshape back to `(batch_questions, 5)` scores -- at that point it's exactly a 5-way
`CrossEntropyLoss` again, just computed from independently-scored options instead of one big
5-option classifier. Gradient clipping and AMP are unchanged from the original notebook.


In [50]:
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

def map3_score(logits, targets):
    top3 = torch.topk(logits, k=3, dim=1).indices
    score = 0.0
    for pred_row, true in zip(top3.tolist(), targets.tolist()):
        if true in pred_row:
            score += 1.0 / (pred_row.index(true) + 1)
    return score / len(targets)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = total_acc = total_map3 = 0.0
    n_batches = 0

    for ids, lengths, y in loader:
        b, n_opts, seq_len = ids.shape                      # n_opts == 5
        ids, lengths, y = ids.to(device), lengths.to(device), y.to(device)
        flat_ids, flat_lengths = ids.view(b * n_opts, seq_len), lengths.view(b * n_opts)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                flat_scores, _ = model(flat_ids, flat_lengths)
                logits = flat_scores.view(b, n_opts)         # (batch, 5) -- back to per-question scores
                loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()

        preds = logits.argmax(dim=1)
        total_loss += loss.item()
        total_acc  += (preds == y).float().mean().item()
        total_map3 += map3_score(logits.detach(), y)
        n_batches  += 1

    return total_loss / n_batches, total_acc / n_batches, total_map3 / n_batches


## 11. Validation + early stopping
**Unchanged pattern** (`model.eval()`, `torch.no_grad()`), but with the loosened patience /
min_delta from `CONFIG` -- gives the model more room for the small, slow late-training gains
BiLSTMs tend to show, instead of stopping the moment improvements drop below a strict threshold.


In [51]:
best_val_map3 = 0.0
patience_left = CONFIG["early_stop_patience"]

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc, train_map3 = run_epoch(train_loader, train=True)
    val_loss,   val_acc,   val_map3   = run_epoch(val_loader,   train=False)
    scheduler.step(val_map3)

    wandb.log({
        "epoch": epoch, "train/loss": train_loss, "train/map3": train_map3,
        "val/loss": val_loss, "val/map3": val_map3, "lr": optimizer.param_groups[0]["lr"],
    })

    if val_map3 > best_val_map3 + CONFIG["early_stop_min_delta"]:
        best_val_map3 = val_map3
        patience_left = CONFIG["early_stop_patience"]
        torch.save(model.state_dict(), "best_bilstm_pairwise.pt")
    elif epoch >= CONFIG["early_stop_min_epochs"]:
        patience_left -= 1

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} val_loss {val_loss:.4f} | val_map3 {val_map3:.4f} | patience {patience_left}")
    if epoch >= CONFIG["early_stop_min_epochs"] and patience_left <= 0:
        print("Early stopping: no val_map3 improvement for", CONFIG["early_stop_patience"], "epochs (after warmup).")
        break

print("Best val MAP@3:", best_val_map3)


Epoch 01 | train_loss 1.2198 val_loss 0.5972 | val_map3 0.9255 | patience 10
Epoch 02 | train_loss 0.4861 val_loss 0.3764 | val_map3 0.9625 | patience 10
Epoch 03 | train_loss 0.3460 val_loss 0.3236 | val_map3 0.9750 | patience 10
Epoch 04 | train_loss 0.3194 val_loss 0.3012 | val_map3 0.9797 | patience 10
Epoch 05 | train_loss 0.2963 val_loss 0.2942 | val_map3 0.9786 | patience 10
Epoch 06 | train_loss 0.2841 val_loss 0.2878 | val_map3 0.9786 | patience 10
Epoch 07 | train_loss 0.2800 val_loss 0.2799 | val_map3 0.9786 | patience 10
Epoch 08 | train_loss 0.2738 val_loss 0.2759 | val_map3 0.9797 | patience 10
Epoch 09 | train_loss 0.2636 val_loss 0.2697 | val_map3 0.9797 | patience 10
Epoch 10 | train_loss 0.2610 val_loss 0.2730 | val_map3 0.9797 | patience 10
Epoch 11 | train_loss 0.2578 val_loss 0.2724 | val_map3 0.9786 | patience 10
Epoch 12 | train_loss 0.2569 val_loss 0.2737 | val_map3 0.9797 | patience 9
Epoch 13 | train_loss 0.2547 val_loss 0.2710 | val_map3 0.9786 | patience 8
E

## 12. MAP@3 inference
**Unchanged logic** (`topk(logits, k=3)`), applied to the reshaped `(batch, 5)` scores coming
out of the pairwise scorer -- from here on it behaves exactly like the original notebook.


In [52]:
model.load_state_dict(torch.load("best_bilstm_pairwise.pt"))
model.eval()

all_top3 = []
with torch.no_grad():
    for ids, lengths in test_loader:
        b, n_opts, seq_len = ids.shape
        ids, lengths = ids.to(device), lengths.to(device)
        flat_scores, _ = model(ids.view(b * n_opts, seq_len), lengths.view(b * n_opts))
        logits = flat_scores.view(b, n_opts)
        top3 = torch.topk(logits, k=3, dim=1).indices.cpu().tolist()
        all_top3.extend(top3)

pred_labels = [" ".join(LABELS[i] for i in row) for row in all_top3]
submission = pd.DataFrame({"ID": test_df["id"], "Prediction": pred_labels})
submission.to_csv("submission_bilstm_pairwise.csv", index=False)
submission.head()


,ID,Prediction
0,1,A D E
1,2,B A E
2,3,B E C
3,4,E C A
4,5,C A B


In [53]:
wandb.finish()


epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
lr,███████▄▄▄▄▂▂▂▂▁▁▁▁▁▁
train/loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/map3,▁▇███████████████████
val/loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/map3,▁▆▇██████████████████
epoch,21
lr,6e-05
train/loss,0.25253
train/map3,0.98839
val/loss,0.27875
